# portfolio-construction_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (131).ipynb`

| Field | Value |
|---|---|
| Section | `portfolio_construction` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 14 |
| Detected functions | daily_correlations, daily_correlations |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Now you can load the entire rows
data = pd.read_parquet('B&F_OCT_2024.parquet')
data

In [ ]:
data.Ticker.unique()

In [ ]:
import pandas as pd

# Ensure Date_Time is in datetime format
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Set Date_Time as the index
data.set_index('Date_Time', inplace=True)

# Add a 'Date' column for grouping by day
data['Date'] = data.index.date

# Pivot the data to have Ticker as columns and BuyPrice as values
pivoted_data = data.pivot_table(index='Date_Time', columns='Ticker', values='BuyPrice')

# Function to calculate daily correlations
def daily_correlations(df):
    # Group by each day and calculate correlations
    return df.groupby(df.index.date).apply(lambda x: x.corr())

# Get daily correlations
daily_corrs = daily_correlations(pivoted_data)

# Reset index to make the daily correlation data easier to work with
daily_corrs = daily_corrs.reset_index(level=0).rename(columns={'level_0': 'Date'})

# Round the correlation values to 2 decimal places
daily_corrs = daily_corrs.round(2)

# Save to CSV
daily_corrs.to_csv('daily_B&T_correlations.csv', index=False)

# Displaying a small part of the output for visualization
print(daily_corrs.head())


In [ ]:
import pandas as pd

# Ensure Date_Time is in datetime format
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Set Date_Time as the index
data.set_index('Date_Time', inplace=True)

# Add a 'Date' column for grouping by day
data['Date'] = data.index.date

# Pivot the data to have Ticker as columns and BuyPrice as values
pivoted_data = data.pivot_table(index='Date_Time', columns='Ticker', values='BuyPrice')

# Function to calculate daily correlations
def daily_correlations(df):
    # Group by each day and calculate correlations
    return df.groupby(df.index.date).apply(lambda x: x.corr())

# Get daily correlations
daily_corrs = daily_correlations(pivoted_data)

# Round the correlation values to 2 decimal places
daily_corrs = daily_corrs.round(2)

# Get negative correlations
negative_corr_pairs = []

# Iterate through each day's correlation matrix
for date, corr_matrix in daily_corrs.groupby(level=0):
    corr_matrix = corr_matrix.droplevel(0)  # Remove the date level
    for stock1 in corr_matrix.columns:
        for stock2 in corr_matrix.columns:
            if stock1 != stock2 and corr_matrix.loc[stock1, stock2] > 0.9:
                negative_corr_pairs.append((date, stock1, stock2, corr_matrix.loc[stock1, stock2]))

# Convert to DataFrame for better representation
negative_corr_df = pd.DataFrame(
    negative_corr_pairs, columns=['Date', 'Stock1', 'Stock2', 'Correlation']
)

# Save to CSV
negative_corr_df.to_csv('negative_correlations.csv', index=False)

# Display the result
print(negative_corr_df)


In [ ]:
import pandas as pd

# Convert Date_Time to datetime
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Define filtering criteria
start_date = '2024-10-01 09:15:00'
end_date = '2024-10-31 15:31:00'
tickers_to_include = ['KOTAKBANK.NSE']

# Apply filtering
filtered_data = data[
    (data['Date_Time'] >= start_date) &
    (data['Date_Time'] <= end_date) &
    (data['Ticker'].isin(tickers_to_include))
]

# Print the filtered dataset with all columns
filtered_data


In [ ]:
import pandas as pd

# Define filtering criteria
start_date = '2024-10-01 09:15:00'
end_date = '2024-10-31 15:31:00'
tickers_to_include = ['AXISBANK.NSE']

# Apply filtering
filtered_data2 = data[
    (data['Date_Time'] >= start_date) &
    (data['Date_Time'] <= end_date) &
    (data['Ticker'].isin(tickers_to_include))
]

# Print the filtered dataset with all columns
filtered_data2


In [ ]:
import pandas as pd

# Assuming df1 and df2 are your two DataFrames and 'Date_Time' is the common column
combined_df = pd.merge(filtered_data2, filtered_data, on='Date_Time', how='inner')  # 'inner' join will keep only the rows where 'Date_Time' is common in both datasets
combined_df.head()


In [ ]:
import pandas as pd

# Assuming combined_df is already defined and 'data' DataFrame is created
data = pd.DataFrame({
    'DateTime': combined_df['Date_Time'],
    'BuyPrice_Stock1': combined_df['BuyPrice_y'],
    'BuyPrice_Stock2': combined_df['BuyPrice_x'],
    'SellPrice_Stock1': combined_df['SellPrice_y'],
    'SellPrice_Stock2': combined_df['SellPrice_x']
})

# Remove rows where any price column has a 0 value
data = data[(data[['BuyPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock1', 'SellPrice_Stock2']] != 0).all(axis=1)]

# Display the resulting DataFrame
data


In [ ]:
import matplotlib.pyplot as plt

# Create the figure
fig = plt.figure(figsize=(13, 7))

# Calculate the difference between the 'close' prices of BuyPrice_Stock1.NS and BuyPrice_Stock2.NS
price_difference_Buy = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) - (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])

# Plot the difference
plt.plot(price_difference_Buy, label='Buy Price Difference')

plt.legend()
plt.title("Difference in Prices: Stock2 - Stock1")
plt.xlabel("Date")
plt.ylabel("Price Difference")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate the ratio of BuyPrice_Stock1 to BuyPrice_Stock2
ratio = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) - (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])

# Calculate the 43-min moving average
moving_average_43 = ratio.rolling(window=6000).mean()

# Calculate the mean and standard deviation of the ratio
mean_ratio = ratio.mean()
std_ratio = ratio.std()

# Calculate the Z-score using the ratio
z_score = (ratio - moving_average_43) / moving_average_43.std()

# Create the figure and plot the Z-score
fig = plt.figure(figsize=(13, 7))
plt.plot(z_score, label='Z-Score of BuyPrice_Stock1 / BuyPrice_Stock2', color='purple')

# Add horizontal lines for Z-scores of -1 and 1 for reference
plt.axhline(0.3, color='green', linestyle='--', label='Z-Score = 0.3')
plt.axhline(-0.3, color='green', linestyle='--', label='Z-Score = -0.3')


# Add buy and sell markers
buy_signals = z_score[z_score < -0.3]
sell_signals = z_score[z_score > 0.3]

# Mark buy signals
plt.scatter(buy_signals.index, buy_signals, marker='^', color='green', label='Buy Signal', s=100)
# Mark sell signals
plt.scatter(sell_signals.index, sell_signals, marker='v', color='red', label='Sell Signal', s=100)

# Add legend, title, and labels
plt.legend()
plt.title("Z-Score of BuyPrice_Stock1 to BuyPrice_Stock2 Ratio with Buy/Sell Signals")
plt.xlabel("Date")
plt.ylabel("Z-Score")
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Load your dataset and prepare
data['datetime'] = pd.to_datetime(data['DateTime'])
data.set_index('datetime', inplace=True)  # Set datetime as index

# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_window = 6000  # Moving average window
z_score_threshold = 0.3  # Z-score threshold

# Initialize variables
portfolio_value = initial_capital
current_position = 0
entry_price1, entry_price2 = None, None
entry_datetime = None
cumulative_pnl = 0
tradebook = []

# Calculate the ratio of BuyPrice_Stock1 to BuyPrice_Stock2
data['ratio'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) - (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
data['moving_average_43'] = data['ratio'].rolling(window=ma_window).mean()
data['std_43'] = data['ratio'].rolling(window=ma_window).std()

# Avoid division by zero for Z-score calculation
data['std_43'].replace(0, np.nan, inplace=True)

# Calculate Z-score
data['z_score'] = (data['ratio'] - data['moving_average_43']) / data['std_43']

# Position size calculations
data['max_risk'] = portfolio_value * risk_percentage
data['position_size1_buy'] = (data['max_risk'] / (2 * data['BuyPrice_Stock1'])).round()
data['position_size1_sell'] = (data['max_risk'] / (2 * data['SellPrice_Stock1'])).round()
data['position_size2_buy'] = (data['max_risk'] / (2 * data['BuyPrice_Stock2'])).round()
data['position_size2_sell'] = (data['max_risk'] / (2 * data['SellPrice_Stock2'])).round()

# Loop through the data for trades
for i, row in data.iterrows():
    sell_price1, buy_price1 = row['SellPrice_Stock1'], row['BuyPrice_Stock1']
    sell_price2, buy_price2 = row['SellPrice_Stock2'], row['BuyPrice_Stock2']
    z_score = row['z_score']
    current_datetime = row.name  # Index as datetime

    # Skip NaN prices or invalid Z-scores
    if pd.isna(sell_price1) or pd.isna(buy_price2) or pd.isna(z_score):
        continue

    # Entry conditions
    if z_score > z_score_threshold and current_position == 0:  # Short stock1, long stock2
        entry_price1, entry_price2 = sell_price1, buy_price2
        entry_datetime = current_datetime
        current_position = -1

    elif z_score < -z_score_threshold and current_position == 0:  # Long stock1, short stock2
        entry_price1, entry_price2 = buy_price1, sell_price2
        entry_datetime = current_datetime
        current_position = 1

    # Exit conditions
    elif current_position != 0 and (abs(z_score) < 0.01 or (current_datetime.hour == 15 and current_datetime.minute >= 30)):
        # Determine exit prices and position sizes
        position_size1 = row['position_size1_sell'] if current_position == -1 else row['position_size1_buy']
        position_size2 = row['position_size2_buy'] if current_position == -1 else row['position_size2_sell']
        exit_price1 = buy_price1 if current_position == -1 else sell_price1
        exit_price2 = sell_price2 if current_position == -1 else buy_price2

        # Calculate PnL and transaction costs
        pnl = (position_size1 * (exit_price1 - entry_price1)) + (position_size2 * (entry_price2 - exit_price2))
        transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) +
                                            position_size2 * (entry_price2 + exit_price2))
        pnl -= transaction_cost
        cumulative_pnl += pnl

        # Record trade details
        tradebook.append({
            'Entry Price Stock1': entry_price1,
            'Entry Price Stock2': entry_price2,
            'Exit Price Stock1': exit_price1,
            'Exit Price Stock2': exit_price2,
            'Trade Type': 'Short' if current_position == -1 else 'Long',
            'Position Size Stock1': position_size1,
            'Position Size Stock2': position_size2,
            'PnL': round(pnl, 2),
            'Transaction Cost': round(transaction_cost, 2),
            'Portfolio Value': portfolio_value,
            'Cumulative PnL': round(cumulative_pnl, 2),
            'Entry Date': entry_datetime.date(),
            'Entry Time': entry_datetime.time(),
            'Exit Date': current_datetime.date(),
            'Exit Time': current_datetime.time(),
            'Time Difference (minutes)': (current_datetime - entry_datetime).total_seconds() / 60
        })

        # Reset position
        current_position = 0

# Convert tradebook to DataFrame and save
tradebook_df = pd.DataFrame(tradebook)
tradebook_df.to_csv('tradebook.csv', index=False)

print(tradebook_df)


In [ ]:
import pandas as pd
import numpy as np

# Load your dataset (replace this with your actual data loading method)
# Ensure your DataFrame has the columns: 'datetime', 'BuyPrice_Stock1', 'SellPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock2'
data['datetime'] = pd.to_datetime(data['DateTime'])

# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_windows = [3000, 4500, 6000, 7500]  # Moving average windows
z_score_thresholds = [0.2, 0.25, 0.3]  # Z-score thresholds

# Results storage
results = []

# Loop through each combination of MA and Z-score threshold
for ma_window in ma_windows:
    # Calculate rolling statistics for Z-score
    data['spread'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) - (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
    data['rolling_mean'] = data['spread'].rolling(window=ma_window).mean()
    data['rolling_std'] = data['spread'].rolling(window=ma_window).std()
    data['z_score'] = (data['spread'] - data['rolling_mean']) / data['rolling_std']

    # Precompute position sizes
    data['max_risk'] = initial_capital * risk_percentage
    for col, price_col in zip(['position_size1_buy', 'position_size1_sell', 'position_size2_buy', 'position_size2_sell'], 
                              ['BuyPrice_Stock1', 'SellPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock2']):
        data[col] = (data['max_risk'] / (2 * data[price_col])).round()

    # Iterate through Z-score thresholds
    for z_threshold in z_score_thresholds:
        # Initialize variables for backtesting
        portfolio_value = initial_capital
        peak_portfolio_value = initial_capital
        max_drawdown = 0
        current_position = 0
        cumulative_pnl = 0
        tradebook = []

        # Iterate through the dataset row-wise
        for _, row in data.iterrows():
            z_score = row['z_score']
            sell_price1, buy_price1 = row['SellPrice_Stock1'], row['BuyPrice_Stock1']
            sell_price2, buy_price2 = row['SellPrice_Stock2'], row['BuyPrice_Stock2']
            current_datetime = row['datetime']

            # Skip iteration if prices are NaN
            if pd.isna([sell_price1, buy_price1, sell_price2, buy_price2]).any():
                continue

            # Entry logic
            if z_score > z_threshold and current_position == 0:  # Short stock1, long stock2
                entry_price1, entry_price2 = sell_price1, buy_price2
                entry_datetime = current_datetime
                current_position = -1

            elif z_score < -z_threshold and current_position == 0:  # Long stock1, short stock2
                entry_price1, entry_price2 = buy_price1, sell_price2
                entry_datetime = current_datetime
                current_position = 1

            # Exit logic
            elif current_position != 0 and (abs(z_score) < 0.01 or (current_datetime.hour == 15 and current_datetime.minute >= 30)):
                # Determine position sizes and exit prices
                position_size1 = row['position_size1_sell'] if current_position == -1 else row['position_size1_buy']
                position_size2 = row['position_size2_buy'] if current_position == -1 else row['position_size2_sell']
                exit_price1 = buy_price1 if current_position == -1 else sell_price1
                exit_price2 = sell_price2 if current_position == -1 else buy_price2

                # Calculate PnL and transaction costs
                pnl = (position_size1 * (exit_price1 - entry_price1)) + (position_size2 * (entry_price2 - exit_price2))
                transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) +
                                                    position_size2 * (entry_price2 + exit_price2))
                pnl -= transaction_cost
                cumulative_pnl += pnl
                portfolio_value += pnl

                # Update drawdown
                peak_portfolio_value = max(peak_portfolio_value, portfolio_value)
                drawdown = (peak_portfolio_value - portfolio_value) / peak_portfolio_value
                max_drawdown = max(max_drawdown, drawdown)

                # Record trade
                tradebook.append({
                    'Entry DateTime': entry_datetime,
                    'Exit DateTime': current_datetime,
                    'Position Size Stock1': position_size1,
                    'Position Size Stock2': position_size2,
                    'PnL': pnl,
                    'Portfolio Value': portfolio_value,
                    'Transaction Cost': transaction_cost
                })

                # Reset position
                current_position = 0

        # Store results for the current configuration
        results.append({
            'Moving Average Window': ma_window,
            'Z-score Threshold': z_threshold,
            'Total PnL': cumulative_pnl,
            'Total Trades': len(tradebook),
            'Final Portfolio Value': portfolio_value,
            'Max Drawdown (%)': round(max_drawdown * 100, 2),
        })

# Convert results to DataFrame and export
results_df = pd.DataFrame(results)
results_df.to_csv('robustness_test_results_AXISBANK_KOTAKBANK.csv', index=False)

# Display results
print(results_df)
